# Task 2 — Notebook 05: Areal statistics + run bundle

**Footprint vs native CDL:** Areas below use **one analysis-grid pixel** = GeoTIFF cell (side length ≈ √(`pixel_area_m2`); often **~320 m** for this stack). That yields **~10 ha/pixel** — a **regional cropland footprint on the analysis grid**, not a USDA acreage estimate from native **30 m** CDL.

Per-class **ha** = valid pixel count × `pixel_area_ha`. A companion **`areal_stats_metadata.json`** next to the CSV records CRS, transform-derived `pixel_area_ha`, and this disclaimer for reviewers.


In [1]:
import json
import sys
import uuid
from datetime import date, datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
import yaml
from pyproj import Transformer
from rasterio.transform import Affine, xy as rio_xy

_cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "requirements.txt").is_file() and (p / "src").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not find repo root.")
sys.path.insert(0, str(REPO_ROOT))

from src.io.cdl_parquet import load_cdl_spatial_metadata

with open(REPO_ROOT / "configs" / "task2_crop_rotation.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

sm_tif = REPO_ROOT / cfg["output"]["processed_dir"] / "rotation_class_map_smoothed.tif"
if not sm_tif.is_file():
    sm_tif = REPO_ROOT / cfg["output"]["processed_dir"] / "rotation_class_map.tif"

with rasterio.open(sm_tif) as src:
    flat = src.read(1).ravel()
    tr = src.transform
    crs_wkt = src.crs.to_wkt() if src.crs else None
    px_w, px_h = tr.a, tr.e
    pixel_area_m2 = abs(px_w * px_h)

pixel_area_ha = pixel_area_m2 / 10000.0
approx_res_m = float(np.sqrt(pixel_area_m2))
names = {0: "regular_rotation", 1: "monoculture", 2: "irregular", 255: "nodata"}
valid = flat[flat != 255]
rows = []
for cls in (0, 1, 2):
    cnt = int(np.sum(valid == cls))
    ha = cnt * pixel_area_ha
    pct = 100.0 * cnt / len(valid) if len(valid) else 0.0
    rows.append(
        {"rotation_class": names[cls], "pixel_count": cnt, "area_ha": round(ha, 1), "pct_of_valid": round(pct, 2)}
    )

overall_df = pd.DataFrame(rows)
try:
    print(overall_df.to_markdown(index=False))
except Exception:
    print(overall_df)

tbl_dir = REPO_ROOT / cfg["output"]["tables_dir"]
tbl_dir.mkdir(parents=True, exist_ok=True)
date_s = date.today().strftime("%Y%m%d")
csv_path = tbl_dir / f"task2__areal_stats_by_class__{date_s}.csv"
overall_df.to_csv(csv_path, index=False)
meta_path = tbl_dir / f"task2__areal_stats_by_class__{date_s}__metadata.json"
meta_doc = {
    "pixel_area_ha": round(pixel_area_ha, 4),
    "pixel_area_m2": round(pixel_area_m2, 2),
    "approx_grid_resolution_m": round(approx_res_m, 1),
    "crs_wkt": crs_wkt,
    "source_geotiff": str(sm_tif.relative_to(REPO_ROOT)).replace("\\", "/"),
    "note": (
        "Areas are total classified cropland footprint on the analysis grid (coarse cells), "
        "aggregated from native 30 m USDA CDL — not a direct USDA NASS acreage comparison."
    ),
    "companion_csv": str(csv_path.relative_to(REPO_ROOT)).replace("\\", "/"),
}
meta_path.write_text(json.dumps(meta_doc, indent=2), encoding="utf-8")
print("Wrote", csv_path.relative_to(REPO_ROOT))
print("Wrote", meta_path.relative_to(REPO_ROOT))

cls_pq = REPO_ROOT / cfg["output"]["processed_dir"] / "rotation_metrics_classified.parquet"
meta_sp = load_cdl_spatial_metadata(REPO_ROOT)
tlist = meta_sp["transform"]
aff = Affine(tlist[0], tlist[1], tlist[2], tlist[3], tlist[4], tlist[5])
crs_s = meta_sp.get("crs") or "EPSG:5070"
mdf = pd.read_parquet(cls_pq)
xs, ys = rio_xy(aff, mdf["iy"].to_numpy(), mdf["ix"].to_numpy(), offset="center")
try:
    trf = Transformer.from_crs(crs_s, "EPSG:4326", always_xy=True)
    lons, lats = trf.transform(xs, ys)
except Exception as exc:
    print("State split skipped (CRS transform):", exc)
    lons = None

reg_csv = None
region_rows = []
if lons is not None:
    joined = None
    try:
        import geopandas as gpd

        states_dir = REPO_ROOT / "data" / "external" / "states"
        shp = list(states_dir.glob("*.shp")) if states_dir.exists() else []
        if shp:
            st = gpd.read_file(shp[0])
            if "NAME" in st.columns:
                st = st[st["NAME"].isin({"Iowa", "Nebraska"})].to_crs(crs_s)
                pts = gpd.GeoDataFrame(mdf, geometry=gpd.points_from_xy(xs, ys), crs=crs_s)
                joined = gpd.sjoin(pts, st[["NAME", "geometry"]], how="left", predicate="within")
                joined["region"] = joined["NAME"].fillna("other")
    except Exception as exc:
        print("State polygon split fallback:", exc)
        joined = None

    if joined is None or "region" not in joined.columns:
        mdf = mdf.copy()
        mdf["region"] = np.where(np.asarray(lons) < -96.35, "Nebraska_proxy_west", "Iowa_proxy_east")
        joined = mdf
        print("Using longitude proxy split at 96.35°W (document in report if no state file).")

    for reg, g in joined.groupby("region"):
        vc = g["rotation_class"].value_counts(normalize=True)
        n = len(g)
        region_rows.append({
            "region": reg,
            "n_pixels": n,
            "pct_regular": round(100 * float(vc.get(0, 0)), 2),
            "pct_monoculture": round(100 * float(vc.get(1, 0)), 2),
            "pct_irregular": round(100 * float(vc.get(2, 0)), 2),
        })
    reg_df = pd.DataFrame(region_rows).sort_values("region")
    try:
        print(reg_df.to_markdown(index=False))
    except Exception:
        print(reg_df)
    reg_csv = tbl_dir / f"task2__areal_stats_by_region__{date_s}.csv"
    reg_df.to_csv(reg_csv, index=False)
    print("Wrote", reg_csv.relative_to(REPO_ROOT))

run_id = uuid.uuid4().hex[:8]
run_dir = REPO_ROOT / "artifacts" / "logs" / "runs" / run_id
run_dir.mkdir(parents=True, exist_ok=True)
bundle = {
    "task": "task2_crop_rotation",
    "run_id": run_id,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "config_path": "configs/task2_crop_rotation.yaml",
    "outputs": {
        "rotation_metrics": str((REPO_ROOT / cfg["output"]["processed_dir"] / "rotation_metrics.parquet").relative_to(REPO_ROOT)).replace("\\", "/"),
        "rotation_map_smoothed": str(sm_tif.relative_to(REPO_ROOT)).replace("\\", "/"),
        "areal_stats_csv": str(csv_path.relative_to(REPO_ROOT)).replace("\\", "/"),
        "areal_stats_metadata": str(meta_path.relative_to(REPO_ROOT)).replace("\\", "/"),
    },
}
if reg_csv is not None:
    bundle["outputs"]["areal_stats_by_region_csv"] = str(reg_csv.relative_to(REPO_ROOT)).replace("\\", "/")
(run_dir / "run_bundle.json").write_text(json.dumps(bundle, indent=2), encoding="utf-8")
print("Wrote", (run_dir / "run_bundle.json").relative_to(REPO_ROOT))


| rotation_class   |   pixel_count |          area_ha |   pct_of_valid |
|:-----------------|--------------:|-----------------:|---------------:|
| regular_rotation |         52080 | 532284           |          17.27 |
| monoculture      |         83068 | 848996           |          27.55 |
| irregular        |        166337 |      1.70005e+06 |          55.17 |
Wrote artifacts\tables\task2\task2__areal_stats_by_class__20260411.csv
Wrote artifacts\tables\task2\task2__areal_stats_by_class__20260411__metadata.json
Using longitude proxy split at 96.35°W (document in report if no state file).
| region              |   n_pixels |   pct_regular |   pct_monoculture |   pct_irregular |
|:--------------------|-----------:|--------------:|------------------:|----------------:|
| Nebraska_proxy_west |     301485 |         16.48 |             26.46 |           57.06 |
Wrote artifacts\tables\task2\task2__areal_stats_by_region__20260411.csv
Wrote artifacts\logs\runs\8d816202\run_bundle.json
